# 💧 LFM2 Inference with Ollama

This notebook demonstrates how to use the [Ollama](https://ollama.com) API to run [LFM2](https://huggingface.co/collections/LiquidAI/lfm2-67d775f3b4b6fe79fbb21bda) and [LFM2.5](https://huggingface.co/collections/LiquidAI/lfm25-6839e3e26b2a9fdbde95b341) models.

> ⚠️ **Note:** Ollama requires local installation on your machine. This notebook shows the Python and curl API usage once Ollama is running locally. Install Ollama from [ollama.com/download](https://ollama.com/download).

## Prerequisites

Before running this notebook, ensure Ollama is installed and running locally:

```bash
# Install Ollama (macOS/Linux)
curl -fsSL https://ollama.com/install.sh | sh

# Pull and run an LFM2 model
ollama run hf.co/LiquidAI/LFM2.5-1.2B-Instruct-GGUF
```

Ollama runs a server on `http://localhost:11434` by default.

## Python Client (OpenAI-compatible)

Use the OpenAI Python client to interact with Ollama:

In [ ]:
!pip install openai -q

In [ ]:
from openai import OpenAI

# Connect to local Ollama server
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="not-needed"
)

# Chat completion
response = client.chat.completions.create(
    model="hf.co/LiquidAI/LFM2.5-1.2B-Instruct-GGUF",
    messages=[
        {"role": "user", "content": "What is C. elegans?"}
    ],
    temperature=0.7,
    max_tokens=512
)
print(response.choices[0].message.content)

## Streaming Responses

In [ ]:
stream = client.chat.completions.create(
    model="hf.co/LiquidAI/LFM2.5-1.2B-Instruct-GGUF",
    messages=[
        {"role": "user", "content": "Tell me a story about space exploration."}
    ],
    stream=True
)

for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="", flush=True)

## Curl Examples

You can also use curl to interact with the Ollama API:

In [ ]:
# Chat API
!curl -s http://localhost:11434/api/chat -d '{
  "model": "hf.co/LiquidAI/LFM2.5-1.2B-Instruct-GGUF",
  "messages": [{"role": "user", "content": "What is machine learning?"}],
  "stream": false
}' | python -c "import sys, json; print(json.load(sys.stdin)['message']['content'])"

In [ ]:
# Generate API (simple completion)
!curl -s http://localhost:11434/api/generate -d '{
  "model": "hf.co/LiquidAI/LFM2.5-1.2B-Instruct-GGUF",
  "prompt": "What is artificial intelligence?",
  "stream": false
}' | python -c "import sys, json; print(json.load(sys.stdin)['response'])"

## Vision Models

LFM2-VL models support image inputs:

In [ ]:
import base64
import requests

# Download and encode image
image_url = "https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg"
img_data = requests.get(image_url).content
image_base64 = base64.b64encode(img_data).decode("utf-8")

# Vision model chat completion
response = client.chat.completions.create(
    model="hf.co/LiquidAI/LFM2.5-VL-1.6B-GGUF",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}},
                {"type": "text", "text": "What's in this image?"}
            ]
        }
    ]
)
print(response.choices[0].message.content)

## Resources

- [LFM2 Documentation](https://docs.liquid.ai/docs/inference/ollama)
- [LFM2 GGUF Models on Hugging Face](https://huggingface.co/LiquidAI/LFM2.5-1.2B-Instruct-GGUF)
- [Ollama Documentation](https://ollama.com)